# MD-DySTGCN — Kaggle GPU v3（改进复现）

相对 v2 的关键改进（本地验证：宗关 RMSE **0.427** / MAPE **8.05%**；持续性基线 0.468；论文 0.265 / 7.67%）：

1. **残差解码**：预测 Δ + 最后观测
2. **水质塔通道独立 TCN**（PatchTST 风格）
3. **时间 last + mean 池化**
4. **L_tcn=6 / L_g=4**
5. **按 val 宗关 RMSE 早停**
6. **宗关 Huber 加权** + Cosine `eta_min=1e-5`

**路径**
- 数据：`/kaggle/input/datasets/bearawa/hanjiang-md-dystgcn-arrays`
- 输出：`/kaggle/working/`

Accelerator 选 **GPU** → Run All。

> 需要：`train.npz` / `val.npz` / `test.npz` / `A_mask.npy` / `scaler_stats.json`


In [ ]:
# ===== 0. Setup =====
import json, math, time, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(42)

# Kaggle dataset path (add dataset: bearawa/hanjiang-md-dystgcn-arrays)
DATA_DIR = Path("/kaggle/input/datasets/bearawa/hanjiang-md-dystgcn-arrays")
# fallback: some notebooks mount under /kaggle/input/<slug>
if not (DATA_DIR / "train.npz").exists():
    cands = sorted(Path("/kaggle/input").rglob("train.npz"))
    if cands:
        DATA_DIR = cands[0].parent
        print("auto DATA_DIR =", DATA_DIR)

OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

need = ["train.npz", "val.npz", "test.npz", "A_mask.npy", "scaler_stats.json"]
missing = [n for n in need if not (DATA_DIR / n).exists()]
if missing:
    raise FileNotFoundError(f"Missing {missing} under {DATA_DIR}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DATA_DIR={DATA_DIR}")
print(f"OUT_DIR={OUT_DIR}")
print(f"DEVICE={DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 1. Config（v3）


In [ ]:
CFG = dict(
    T_in=168,
    T_out=12,
    N=16,
    C=9,
    D=4,
    F=128,
    d_E=64,
    mlp_hidden=128,
    L_tcn=6,          # dilations 1..32
    K_tcn=3,
    L_g=4,
    dropout=0.1,
    batch_size=64,
    lr=1e-3,
    weight_decay=1e-4,
    huber_delta=1.0,
    epochs=60,
    patience=15,
    zongguan_idx=15,
    num_workers=2,    # Kaggle 建议 2
    use_amp=True,
    use_cosine=True,
    seed=42,
    zongguan_loss_weight=4.0,
    residual=True,
)
# 冒烟可改：CFG["epochs"] = 2
print(CFG)


## 2. Dataset


In [ ]:
class NPZDataset(Dataset):
    def __init__(self, path: Path):
        d = np.load(path)
        self.X = torch.from_numpy(d["X"].astype(np.float32))
        self.M = torch.from_numpy(d["M"].astype(np.float32))
        self.Y = torch.from_numpy(d["Y"].astype(np.float32))

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        return self.X[i], self.M[i], self.Y[i]


scaler = json.loads((DATA_DIR / "scaler_stats.json").read_text())
WATER_MU = np.asarray(scaler["water_mu"], dtype=np.float64)
WATER_STD = np.asarray(scaler["water_std"], dtype=np.float64)
FEATS = scaler["water_features"]

A_mask = torch.from_numpy(np.load(DATA_DIR / "A_mask.npy").astype(np.float32)).to(DEVICE)
print("A_mask", tuple(A_mask.shape), "sum", float(A_mask.sum()))

kw = dict(
    batch_size=CFG["batch_size"],
    num_workers=CFG["num_workers"],
    pin_memory=DEVICE.type == "cuda",
)
train_loader = DataLoader(NPZDataset(DATA_DIR / "train.npz"), shuffle=True, drop_last=True, **kw)
val_loader = DataLoader(NPZDataset(DATA_DIR / "val.npz"), shuffle=False, **kw)
test_loader = DataLoader(NPZDataset(DATA_DIR / "test.npz"), shuffle=False, **kw)
print(f"train={len(train_loader.dataset)} val={len(val_loader.dataset)} test={len(test_loader.dataset)}")

# persistence baseline
te = np.load(DATA_DIR / "test.npz")
Yte = te["Y"][:, :, CFG["zongguan_idx"], :]
Xte = te["X"][:, :, CFG["zongguan_idx"], :]
persist = np.repeat(Xte[:, -1:, :], Yte.shape[1], axis=1)
P_RMSE = float(np.sqrt(((persist - Yte) ** 2).mean()))
print(f"Persistence RMSE(zg)={P_RMSE:.4f}  paper=0.2649")


## 3. Model（v3：通道独立水质塔 + 残差解码）


In [ ]:
class ResidualTCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, dilation=1, dropout=0.1):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation, padding=pad)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, dilation=dilation, padding=pad)
        self.drop = nn.Dropout(dropout)
        self.proj = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.cut = pad

    def _causal(self, y, T):
        if self.cut > 0:
            y = y[:, :, :-self.cut]
        return y[:, :, :T]

    def forward(self, x):
        T = x.size(2)
        y = self.drop(F.relu(self._causal(self.conv1(x), T)))
        y = self.drop(F.relu(self._causal(self.conv2(y), T)))
        return F.relu(y + self.proj(x))


class TemporalTower(nn.Module):
    """Shared multi-channel TCN for meteo. [B,T,N,D] -> [B,T,N,F]."""
    def __init__(self, in_dim, hidden, n_layers=6, kernel_size=3, dropout=0.1):
        super().__init__()
        layers, ch_in = [], in_dim
        for i in range(n_layers):
            layers.append(ResidualTCNBlock(ch_in, hidden, kernel_size, 2**i, dropout))
            ch_in = hidden
        self.blocks = nn.ModuleList(layers)

    def forward(self, x):
        B, T, N, C = x.shape
        h = x.permute(0, 2, 3, 1).reshape(B * N, C, T)
        for blk in self.blocks:
            h = blk(h)
        return h.reshape(B, N, h.shape[1], T).permute(0, 3, 1, 2)


class ChannelIndepTemporalTower(nn.Module):
    """Channel-independent TCN for water (PatchTST-style)."""
    def __init__(self, n_channels, hidden, n_layers=6, kernel_size=3, dropout=0.1):
        super().__init__()
        layers, ch_in = [], 1
        for i in range(n_layers):
            layers.append(ResidualTCNBlock(ch_in, hidden, kernel_size, 2**i, dropout))
            ch_in = hidden
        self.blocks = nn.ModuleList(layers)
        self.fuse = nn.Sequential(
            nn.Linear(n_channels * hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
        )

    def forward(self, x):
        B, T, N, C = x.shape
        h = x.permute(0, 2, 3, 1).reshape(B * N * C, 1, T)
        for blk in self.blocks:
            h = blk(h)
        Fh = h.shape[1]
        h = h.reshape(B, N, C, Fh, T).permute(0, 4, 1, 2, 3).contiguous()
        return self.fuse(h.reshape(B, T, N, C * Fh))


class DynamicAdjacency(nn.Module):
    def __init__(self, d_meteo, d_E=64, mlp_hidden=128):
        super().__init__()
        self.embed = nn.Linear(d_meteo, d_E)
        self.mlp = nn.Sequential(nn.Linear(1, mlp_hidden), nn.ReLU(), nn.Linear(mlp_hidden, 1))
        self.d_E = d_E

    def forward(self, M, A_mask):
        B, T, N, D = M.shape
        E = F.relu(self.embed(M.reshape(B * T, N, D)))
        S = torch.matmul(E, E.transpose(1, 2)) / math.sqrt(self.d_E)
        A_prime = F.relu(self.mlp(S.unsqueeze(-1)).squeeze(-1))
        mask = A_mask.to(A_prime.device).clone()
        eye = torch.eye(N, device=mask.device, dtype=mask.dtype)
        mask = torch.clamp(mask + eye, max=1.0)
        logits = A_prime.masked_fill(mask.unsqueeze(0) < 0.5, -1e4)
        A = torch.softmax(logits, dim=-1)
        return A.reshape(B, T, N, N)


def normalize_adj(A):
    deg = A.sum(dim=-1).clamp(min=1e-6)
    deg_inv_sqrt = deg.pow(-0.5)
    return deg_inv_sqrt.unsqueeze(-1) * A * deg_inv_sqrt.unsqueeze(-2)


class DynGCNLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.theta = nn.Linear(dim, dim, bias=True)
        self.res = nn.Identity()

    def forward(self, Z, A):
        A_prop = A.transpose(-1, -2)  # downstream aggregates upstream
        A_norm = normalize_adj(A_prop)
        return F.relu(self.theta(torch.matmul(A_norm, Z))) + self.res(Z)


class DynGCN(nn.Module):
    def __init__(self, hidden, n_layers=4):
        super().__init__()
        self.layers = nn.ModuleList([DynGCNLayer(hidden) for _ in range(n_layers)])

    def forward(self, Z, A):
        for layer in self.layers:
            Z = layer(Z, A)
        return Z


class GatedFusion(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.fc = nn.Linear(hidden * 2, hidden)

    def forward(self, Zw, Zm):
        gate = torch.sigmoid(self.fc(torch.cat([Zw, Zm], dim=-1)))
        return gate * Zm + (1.0 - gate) * Zw


class MDDySTGCNv3(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        Fh, drop = cfg["F"], cfg["dropout"]
        self.water_tower = ChannelIndepTemporalTower(
            cfg["C"], Fh, n_layers=cfg["L_tcn"], kernel_size=cfg["K_tcn"], dropout=drop
        )
        self.meteo_tower = TemporalTower(
            cfg["D"], Fh, n_layers=cfg["L_tcn"], kernel_size=cfg["K_tcn"], dropout=drop
        )
        self.dyn_adj = DynamicAdjacency(cfg["D"], d_E=cfg["d_E"], mlp_hidden=cfg["mlp_hidden"])
        self.gcn_w = DynGCN(Fh, n_layers=cfg["L_g"])
        self.gcn_m = DynGCN(Fh, n_layers=cfg["L_g"])
        self.gate = GatedFusion(Fh)
        self.decoder = nn.Sequential(
            nn.Linear(Fh * 2, Fh * 2),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(Fh * 2, cfg["T_out"] * cfg["C"]),
        )

    def forward(self, X, M, A_mask):
        Hw = self.water_tower(X)
        Hm = self.meteo_tower(M)
        A = self.dyn_adj(M, A_mask)
        Zw = self.gcn_w(Hw, A)
        Zm = self.gcn_m(Hm, A)
        H = self.gate(Zw, Zm)
        h = torch.cat([H[:, -1], H.mean(dim=1)], dim=-1)
        out = self.decoder(h)
        B, N, _ = out.shape
        Tout, C = self.cfg["T_out"], self.cfg["C"]
        delta = out.view(B, N, Tout, C).permute(0, 2, 1, 3).contiguous()
        if self.cfg.get("residual", True):
            return delta + X[:, -1, :, :].unsqueeze(1)
        return delta


model = MDDySTGCNv3(CFG).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {n_params:,}")

Xb, Mb, Yb = next(iter(train_loader))
with torch.no_grad():
    yhat = model(Xb.to(DEVICE), Mb.to(DEVICE), A_mask)
print("Yhat", tuple(yhat.shape), "Y", tuple(Yb.shape))


## 4. Loss / Metrics / Train


In [ ]:
def station_weighted_huber(pred, target, cfg):
    per = F.huber_loss(pred, target, delta=cfg["huber_delta"], reduction="none")
    w = torch.ones(cfg["N"], device=pred.device, dtype=per.dtype)
    w[cfg["zongguan_idx"]] = cfg["zongguan_loss_weight"]
    return (per * w.view(1, 1, -1, 1)).mean()


@torch.no_grad()
def eval_loader(model, loader, A_mask, cfg, water_mu, water_std, compute_mape=False):
    model.eval()
    total_loss = n = 0.0
    se = ae = count = 0.0
    mape_num = mape_den = 0.0
    zi = cfg["zongguan_idx"]
    use_amp = cfg["use_amp"] and DEVICE.type == "cuda"
    per_se = np.zeros(cfg["C"], dtype=np.float64)
    per_n = 0

    for X, M, Y in loader:
        X = X.to(DEVICE, non_blocking=True)
        M = M.to(DEVICE, non_blocking=True)
        Y = Y.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            Yhat = model(X, M, A_mask)
            loss = station_weighted_huber(Yhat, Y, cfg)
        bs = X.size(0)
        total_loss += loss.item() * bs
        n += bs
        pred_z = Yhat[:, :, zi, :].float()
        true_z = Y[:, :, zi, :].float()
        se += ((pred_z - true_z) ** 2).sum().item()
        ae += (pred_z - true_z).abs().sum().item()
        count += pred_z.numel()
        diff = (pred_z - true_z).detach().cpu().numpy()
        per_se += (diff ** 2).sum(axis=(0, 1))
        per_n += diff.shape[0] * diff.shape[1]
        if compute_mape:
            pred_p = pred_z.cpu().numpy() * water_std + water_mu
            true_p = true_z.cpu().numpy() * water_std + water_mu
            denom = np.maximum(np.abs(true_p), 1e-3)
            mape_num += np.abs((pred_p - true_p) / denom).sum()
            mape_den += true_p.size

    out = {
        "loss": total_loss / max(n, 1),
        "rmse": math.sqrt(se / max(count, 1)),
        "mae": ae / max(count, 1),
        "mse": se / max(count, 1),
        "per_channel_rmse": (np.sqrt(per_se / max(per_n, 1))).tolist(),
    }
    if compute_mape:
        out["mape"] = 100.0 * mape_num / max(mape_den, 1)
    return out


def train_model(model, train_loader, val_loader, A_mask, cfg, out_dir):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scheduler = (
        torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"], eta_min=1e-5)
        if cfg.get("use_cosine", True) else None
    )
    use_amp = cfg["use_amp"] and DEVICE.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    best_val = float("inf")
    best_path = out_dir / "md_dystgcn_v3_best.pt"
    patience_left = cfg["patience"]
    history = {"train_loss": [], "val_loss": [], "val_rmse": [], "lr": []}

    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        t0 = time.time()
        run_loss, n = 0.0, 0
        for X, M, Y in train_loader:
            X = X.to(DEVICE, non_blocking=True)
            M = M.to(DEVICE, non_blocking=True)
            Y = Y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=use_amp):
                Yhat = model(X, M, A_mask)
                loss = station_weighted_huber(Yhat, Y, cfg)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt)
            scaler.update()
            run_loss += loss.item() * X.size(0)
            n += X.size(0)

        if scheduler is not None:
            scheduler.step()

        train_loss = run_loss / max(n, 1)
        val_m = eval_loader(model, val_loader, A_mask, cfg, WATER_MU, WATER_STD)
        cur_lr = opt.param_groups[0]["lr"]
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_m["loss"])
        history["val_rmse"].append(val_m["rmse"])
        history["lr"].append(cur_lr)
        print(
            f"Epoch {epoch:03d}/{cfg['epochs']}  "
            f"train_loss={train_loss:.4f}  val_loss={val_m['loss']:.4f}  "
            f"val_RMSE(zg)={val_m['rmse']:.4f}  lr={cur_lr:.2e}  ({time.time()-t0:.1f}s)"
        )

        if val_m["rmse"] < best_val - 1e-5:
            best_val = val_m["rmse"]
            patience_left = cfg["patience"]
            torch.save(
                {"model": model.state_dict(), "cfg": cfg, "epoch": epoch,
                 "val_rmse": best_val, "val_metrics": val_m},
                best_path,
            )
            print(f"  -> saved best val_RMSE={best_val:.4f}")
        else:
            patience_left -= 1
            if patience_left <= 0:
                print("Early stopping.")
                break

    with open(out_dir / "history.json", "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)
    return history, best_path

print("train helpers ready")


## 5. Run training


In [ ]:
history, best_path = train_model(model, train_loader, val_loader, A_mask, CFG, OUT_DIR)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(history["train_loss"], label="train")
ax[0].plot(history["val_loss"], label="val")
ax[0].set_title("Huber Loss (zg-weighted)")
ax[0].legend(); ax[0].grid(True, ls=":")
ax[1].plot(history["val_rmse"], color="C1", label="val")
ax[1].axhline(P_RMSE, color="gray", ls="--", label=f"persist {P_RMSE:.3f}")
ax[1].axhline(0.2649, color="green", ls="--", label="paper 0.265")
ax[1].set_title("Val RMSE (Zongguan, z-space)")
ax[1].legend(fontsize=8); ax[1].grid(True, ls=":")
fig.tight_layout()
fig.savefig(OUT_DIR / "loss_curves.png", dpi=140)
plt.show()
print("saved", OUT_DIR / "loss_curves.png")


## 6. Test evaluation（宗关）


In [ ]:
ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model"])
print(f"Loaded best epoch={ckpt.get('epoch')} val_RMSE={ckpt.get('val_rmse'):.4f}")

test_m = eval_loader(model, test_loader, A_mask, CFG, WATER_MU, WATER_STD, compute_mape=True)
per = {f: round(r, 4) for f, r in zip(FEATS, test_m["per_channel_rmse"])}
metrics = {
    "test_loss": test_m["loss"],
    "test_rmse_zongguan": test_m["rmse"],
    "test_mae_zongguan": test_m["mae"],
    "test_mse_zongguan": test_m["mse"],
    "test_mape_zongguan_pct": test_m["mape"],
    "per_channel_rmse_zongguan": per,
    "persistence_rmse": P_RMSE,
    "paper_ref_rmse": 0.2649,
    "paper_ref_mape_pct": 7.67,
    "best_epoch": ckpt.get("epoch"),
    "best_val_rmse": ckpt.get("val_rmse"),
    "n_params": int(n_params),
    "device": str(DEVICE),
    "cfg": CFG,
    "improvements": [
        "residual_decoding",
        "channel_indep_water_tcn",
        "temporal_last+mean_pool",
        "L_tcn=6_RF_cover_Tin",
        "L_g=4_less_oversmooth",
        "early_stop_on_val_rmse",
        "zongguan_loss_weight=4",
        "cosine_eta_min=1e-5",
    ],
}
print(json.dumps({k: v for k, v in metrics.items() if k != "cfg"}, indent=2, ensure_ascii=False))
with open(OUT_DIR / "test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print("wrote", OUT_DIR / "test_metrics.json")
print("checkpoint", best_path)


## 7. 可选：宗关预测曲线（horizon-1 拼接）


In [ ]:
@torch.no_grad()
def collect_h1(model, loader, A_mask, zi):
    model.eval()
    preds, trues = [], []
    use_amp = CFG["use_amp"] and DEVICE.type == "cuda"
    for X, M, Y in loader:
        X = X.to(DEVICE); M = M.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=use_amp):
            Yhat = model(X, M, A_mask)
        preds.append(Yhat[:, 0, zi, :].float().cpu().numpy())
        trues.append(Y[:, 0, zi, :].numpy())
    return np.concatenate(preds), np.concatenate(trues)

pred_z, true_z = collect_h1(model, test_loader, A_mask, CFG["zongguan_idx"])
n = min(200, len(pred_z))
pred = pred_z[:n] * WATER_STD + WATER_MU
true = true_z[:n] * WATER_STD + WATER_MU
names = ["WT", "pH", "DO", "CODMn", "NH3-N", "TP", "TN", "EC", "Turbidity"]

fig, axes = plt.subplots(3, 3, figsize=(14, 10), sharex=True)
x = np.arange(n)
for ax, name, i in zip(axes.ravel(), names, range(9)):
    ax.plot(x, true[:, i], color="black", lw=1.0, label="真实值")
    ax.plot(x, pred[:, i], color="red", ls="--", lw=1.0, label="预测值")
    ax.set_title(f"{name} (站点: 宗关)")
    ax.grid(True, color="0.85", lw=0.7)
    ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "zongguan_forecast.png", dpi=140)
plt.show()
print("saved", OUT_DIR / "zongguan_forecast.png")


## 备注（v3）

| 项 | 说明 |
|----|------|
| 数据 | 挂载 `bearawa/hanjiang-md-dystgcn-arrays` |
| 相对 v2 | 残差解码、通道独立水质 TCN、last+mean、L_tcn=6、L_g=4、RMSE 早停、宗关加权 |
| 本地参考 | test RMSE≈0.427 / MAPE≈8.05%；persist≈0.468；论文 0.265 / 7.67% |
| OOM | 将 `batch_size` 改为 32 |
